# 📊 Electronic Campaigns Efficiency Analysis
**Author:** Abdelrahman Elmogy  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Data:** Synthetic dataset mirroring real campaign structure

---
## Objective
Analyze the performance of multi-channel electronic collection campaigns (IVR, WhatsApp, Email, SMS)
to identify the most effective channels, optimal contact times, and highest-ROI segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)

# Load data
campaigns = pd.read_csv('../data/sample_campaigns.csv')
contacts  = pd.read_csv('../data/sample_contacts.csv', parse_dates=['contact_time'])

print('Campaigns:', campaigns.shape)
print('Contacts: ', contacts.shape)
campaigns.head()

## 1. Channel Performance Overview

In [ ]:
channel_kpis = campaigns.groupby('channel').agg(
    total_campaigns   = ('campaign_id',    'count'),
    avg_delivery_rate = ('delivery_rate',  'mean'),
    avg_response_rate = ('response_rate',  'mean'),
    avg_conversion    = ('conversion_rate','mean'),
    avg_roi           = ('campaign_roi',   'mean'),
    total_recovered   = ('recovery_amount','sum'),
    total_cost        = ('total_cost',     'sum'),
).round(3).reset_index()

print(channel_kpis.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['avg_response_rate', 'avg_conversion', 'avg_roi']
titles  = ['Avg Response Rate', 'Avg Conversion Rate', 'Avg Campaign ROI (%)']

for ax, metric, title in zip(axes, metrics, titles):
    sns.barplot(data=channel_kpis, x='channel', y=metric, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Channel Performance Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/channel_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Monthly Recovery Trend per Channel

In [ ]:
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['month'] = campaigns['start_date'].dt.to_period('M').astype(str)

monthly = campaigns.groupby(['month','channel'])['recovery_amount'].sum().reset_index()

pivot = monthly.pivot(index='month', columns='channel', values='recovery_amount').fillna(0)
pivot.plot(marker='o', figsize=(14, 5))

plt.title('Monthly Recovery Amount by Channel', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Recovery Amount')
plt.xticks(rotation=45)
plt.legend(title='Channel')
plt.tight_layout()
plt.savefig('../visuals/monthly_recovery_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Best Hours to Contact Customers

In [ ]:
contacts['hour'] = contacts['contact_time'].dt.hour

hourly = contacts.groupby(['channel','hour']).agg(
    total    = ('contact_id','count'),
    responded = ('responded','sum')
).reset_index()
hourly['response_rate'] = hourly['responded'] / hourly['total']

g = sns.FacetGrid(hourly, col='channel', col_wrap=2, height=4, sharey=False)
g.map_dataframe(sns.lineplot, x='hour', y='response_rate', marker='o')
g.set_titles(col_template='{col_name}')
g.set_axis_labels('Hour of Day', 'Response Rate')
g.figure.suptitle('Response Rate by Hour per Channel', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/best_contact_hours.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Segment Analysis — Which Segment Converts Best on Each Channel?

In [ ]:
seg_perf = campaigns.groupby(['segment','channel']).agg(
    avg_conversion = ('conversion_rate','mean'),
    avg_roi        = ('campaign_roi',   'mean')
).round(3).reset_index()

pivot_conv = seg_perf.pivot(index='segment', columns='channel', values='avg_conversion')

sns.heatmap(pivot_conv, annot=True, fmt='.2%', cmap='YlGn', linewidths=0.5)
plt.title('Avg Conversion Rate: Segment × Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/segment_channel_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Cost Efficiency — ROI vs. Cost per Contact

In [ ]:
plt.figure(figsize=(10, 6))
for channel in campaigns['channel'].unique():
    subset = campaigns[campaigns['channel'] == channel]
    plt.scatter(subset['cost_per_contact'], subset['campaign_roi'],
                label=channel, alpha=0.6, s=60)

plt.xlabel('Cost per Contact', fontsize=12)
plt.ylabel('Campaign ROI (%)', fontsize=12)
plt.title('ROI vs. Cost per Contact by Channel', fontsize=14, fontweight='bold')
plt.legend(title='Channel')
plt.tight_layout()
plt.savefig('../visuals/roi_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Summary of Key Findings

| Finding | Detail |
|---------|--------|
| Best conversion channel | WhatsApp |
| Lowest cost channel | Email |
| Best contact window | 10 AM – 12 PM |
| Highest ROI segment combo | (See heatmap above) |

> These findings directly informed campaign scheduling and channel allocation strategy.